# Cognitive IDS — Network Anomaly Detection
**Ensemble: BiLSTM + CNN + Transformer + VAE + XGBoost meta-classifier**

Run cells top-to-bottom. GPU runtime recommended (Runtime > Change runtime type > T4 GPU).

In [ ]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Set your project root — change this if you placed the folder elsewhere
PROJECT_ROOT = '/content/drive/MyDrive/cognitive_ids'
import sys, os
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
sys.path.insert(0, PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Files:', os.listdir(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────
!pip install -q imbalanced-learn xgboost seaborn
print('Dependencies installed.')

In [ ]:
# ── Cell 3: GPU check ────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('Memory growth enabled.')
else:
    print('WARNING: No GPU found. Training will be slow on CPU.')

## Step 1 — Configure paths
Edit the paths below to match where you put your dataset files.

In [ ]:
# ── Cell 4: Paths configuration ──────────────────────────────────────────
import os

# NSL-KDD paths
NSL_TRAIN_PATH = os.path.join(PROJECT_ROOT, 'data/NSLKDD/KDDTrain+.txt')
NSL_TEST_PATH  = os.path.join(PROJECT_ROOT, 'data/NSLKDD/KDDTest+.txt')

# CICIDS2017 — folder containing all 8 CSV files
CICIDS_DIR = os.path.join(PROJECT_ROOT, 'data/CICIDS2017/')

# Output directories
SAVE_DIR_NSL   = os.path.join(PROJECT_ROOT, 'data/processed/nslkdd')
SAVE_DIR_CIC   = os.path.join(PROJECT_ROOT, 'data/processed/cicids')
MODELS_DIR     = os.path.join(PROJECT_ROOT, 'models')
OUTPUTS_DIR    = os.path.join(PROJECT_ROOT, 'outputs')

for d in [SAVE_DIR_NSL, SAVE_DIR_CIC, MODELS_DIR, OUTPUTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('NSL train exists:', os.path.exists(NSL_TRAIN_PATH))
print('NSL test  exists:', os.path.exists(NSL_TEST_PATH))
print('CICIDS dir exists:', os.path.exists(CICIDS_DIR))
if os.path.exists(CICIDS_DIR):
    print('CICIDS files:', os.listdir(CICIDS_DIR))

## Step 2 — Preprocess NSL-KDD

In [ ]:
# ── Cell 5: Preprocess NSL-KDD ───────────────────────────────────────────
from preprocess import prepare_nslkdd

X_train_nsl, X_test_nsl, y_train_nsl, y_test_nsl, feat_nsl = prepare_nslkdd(
    train_path   = NSL_TRAIN_PATH,
    test_path    = NSL_TEST_PATH,
    save_dir     = SAVE_DIR_NSL,
    use_smote    = True,
    random_state = 42
)
print(f'NSL-KDD  Train: {X_train_nsl.shape}  Test: {X_test_nsl.shape}')

## Step 3 — Preprocess CICIDS2017

In [ ]:
# ── Cell 6: Preprocess CICIDS2017 ────────────────────────────────────────
from preprocess import prepare_cicids

X_train_cic, X_test_cic, y_train_cic, y_test_cic, feat_cic = prepare_cicids(
    data_dir     = CICIDS_DIR,
    save_dir     = SAVE_DIR_CIC,
    use_smote    = True,
    random_state = 42
)
print(f'CICIDS2017  Train: {X_train_cic.shape}  Test: {X_test_cic.shape}')

## Step 4 — SSA Hyperparameter Optimization
This finds optimal hyperparams for each model. Skip to Step 5 to use defaults.

In [ ]:
# ── Cell 7: SSA optimization (optional — run once, then use best params) ─
import numpy as np
from sklearn.model_selection import train_test_split
from ssa_optimizer import (
    SalpSwarmOptimizer,
    BILSTM_SPACE, CNN_SPACE, TRANSFORMER_SPACE,
    make_bilstm_fitness, make_cnn_fitness, make_transformer_fitness,
    decode_bilstm_params, decode_cnn_params, decode_transformer_params
)

# Use NSL-KDD for optimization (faster than CICIDS)
# Use a small subset to keep SSA tractable
idx = np.random.choice(len(X_train_nsl), min(20000, len(X_train_nsl)), replace=False)
X_opt = X_train_nsl[idx]
y_opt = y_train_nsl[idx]
X_ot, X_ov, y_ot, y_ov = train_test_split(X_opt, y_opt, test_size=0.2,
                                           stratify=y_opt, random_state=42)

# ── BiLSTM SSA ──
print('\n=== SSA for BiLSTM ===')
ssa_bilstm = SalpSwarmOptimizer(
    n_salps=5, max_iter=15,
    lb=BILSTM_SPACE['lb'], ub=BILSTM_SPACE['ub'], dim=BILSTM_SPACE['dim']
)
best_pos_bilstm, _, conv_bilstm = ssa_bilstm.optimize(
    make_bilstm_fitness(X_ot, y_ot, X_ov, y_ov, epochs=10)
)
best_bilstm_params = decode_bilstm_params(best_pos_bilstm)
print('Best BiLSTM params:', best_bilstm_params)

# ── CNN SSA ──
print('\n=== SSA for CNN ===')
ssa_cnn = SalpSwarmOptimizer(
    n_salps=5, max_iter=15,
    lb=CNN_SPACE['lb'], ub=CNN_SPACE['ub'], dim=CNN_SPACE['dim']
)
best_pos_cnn, _, conv_cnn = ssa_cnn.optimize(
    make_cnn_fitness(X_ot, y_ot, X_ov, y_ov, epochs=10)
)
best_cnn_params = decode_cnn_params(best_pos_cnn)
print('Best CNN params:', best_cnn_params)

# ── Transformer SSA ──
print('\n=== SSA for Transformer ===')
ssa_tf = SalpSwarmOptimizer(
    n_salps=5, max_iter=15,
    lb=TRANSFORMER_SPACE['lb'], ub=TRANSFORMER_SPACE['ub'],
    dim=TRANSFORMER_SPACE['dim']
)
best_pos_tf, _, conv_tf = ssa_tf.optimize(
    make_transformer_fitness(X_ot, y_ot, X_ov, y_ov, epochs=10)
)
best_tf_params = decode_transformer_params(best_pos_tf)
print('Best Transformer params:', best_tf_params)

import joblib, os
joblib.dump({'bilstm': best_bilstm_params,
             'cnn':    best_cnn_params,
             'transformer': best_tf_params,
             'conv_bilstm': conv_bilstm,
             'conv_cnn':    conv_cnn,
             'conv_tf':     conv_tf},
            os.path.join(MODELS_DIR, 'ssa_best_params.pkl'))
print('\nBest params saved.')

## Step 5 — Train all models
Uses SSA-tuned params if Cell 7 was run, otherwise uses safe defaults.

In [ ]:
# ── Cell 8: Load best params or use defaults ──────────────────────────────
import joblib, os
ssa_path = os.path.join(MODELS_DIR, 'ssa_best_params.pkl')

if os.path.exists(ssa_path):
    best = joblib.load(ssa_path)
    bilstm_params = best['bilstm']
    cnn_params    = best['cnn']
    tf_params     = best['transformer']
    print('Loaded SSA-tuned parameters.')
else:
    print('SSA params not found — using defaults.')
    bilstm_params = {'lstm_units':128, 'dense_units':64,
                     'dropout_rate':0.3, 'l2_reg':1e-4, 'batch_size':256}
    cnn_params    = {'filters':64,  'dense_units':64,
                     'dropout_rate':0.3, 'l2_reg':1e-4, 'batch_size':256}
    tf_params     = {'embed_dim':64, 'num_heads':4, 'ff_dim':128,
                     'num_blocks':2, 'dropout_rate':0.2}

vae_params = {'latent_dim':16, 'hidden_dim':64, 'dropout':0.2}

print('BiLSTM params:     ', bilstm_params)
print('CNN params:        ', cnn_params)
print('Transformer params:', tf_params)
print('VAE params:        ', vae_params)

In [ ]:
# ── Cell 9: Train on NSL-KDD ──────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from bilstm_model     import train_bilstm
from cnn_model        import train_cnn
from transformer_model import train_transformer
from vae_model        import train_vae

# Split a validation set from training data
X_tr_n, X_vl_n, y_tr_n, y_vl_n = train_test_split(
    X_train_nsl, y_train_nsl, test_size=0.1, stratify=y_train_nsl, random_state=42
)

print('\n===== Training BiLSTM (NSL-KDD) =====')
bilstm_nsl, hist_bilstm_nsl = train_bilstm(
    X_tr_n, y_tr_n, X_vl_n, y_vl_n,
    **{k:v for k,v in bilstm_params.items() if k != 'batch_size'},
    batch_size=bilstm_params.get('batch_size', 256),
    epochs=100,
    save_path=os.path.join(MODELS_DIR, 'bilstm_nsl.keras')
)

print('\n===== Training CNN (NSL-KDD) =====')
cnn_nsl, hist_cnn_nsl = train_cnn(
    X_tr_n, y_tr_n, X_vl_n, y_vl_n,
    **{k:v for k,v in cnn_params.items() if k != 'batch_size'},
    batch_size=cnn_params.get('batch_size', 256),
    epochs=100,
    save_path=os.path.join(MODELS_DIR, 'cnn_nsl.keras')
)

print('\n===== Training Transformer (NSL-KDD) =====')
transformer_nsl, hist_tf_nsl = train_transformer(
    X_tr_n, y_tr_n, X_vl_n, y_vl_n,
    **tf_params,
    epochs=100,
    save_path=os.path.join(MODELS_DIR, 'transformer_nsl.keras')
)

print('\n===== Training VAE (NSL-KDD) =====')
vae_nsl, enc_nsl, dec_nsl, hist_vae_nsl = train_vae(
    X_tr_n, y_tr_n,
    **vae_params,
    epochs=100,
    save_dir=os.path.join(MODELS_DIR, 'vae_nsl')
)
print('\nAll NSL-KDD models trained.')

In [ ]:
# ── Cell 10: Train ensemble on NSL-KDD ───────────────────────────────────
from ensemble import collect_probabilities, EnsembleMetaClassifier

# Collect validation probabilities for fusion weight learning
proba_val_nsl = collect_probabilities(
    bilstm_nsl, cnn_nsl, transformer_nsl, enc_nsl, dec_nsl, X_vl_n
)

ensemble_nsl = EnsembleMetaClassifier()
ensemble_nsl.fit(proba_val_nsl, y_vl_n,
                 model_names=['BiLSTM','CNN','Transformer','VAE'])
ensemble_nsl.save(os.path.join(MODELS_DIR, 'ensemble_nsl'))
print('NSL-KDD ensemble trained and saved.')

In [ ]:
# ── Cell 11: Evaluate on NSL-KDD test set ────────────────────────────────
from evaluate import (
    compute_all_metrics, plot_confusion_matrix,
    plot_roc_curves, plot_metrics_bar, plot_training_history,
    save_metrics_table, full_evaluation
)
from bilstm_model     import predict_proba_bilstm
from cnn_model        import predict_proba_cnn
from transformer_model import predict_proba_transformer
from vae_model        import predict_proba_vae

out_nsl = os.path.join(OUTPUTS_DIR, 'nslkdd')
os.makedirs(out_nsl, exist_ok=True)

# Individual model predictions
p_bilstm  = predict_proba_bilstm(bilstm_nsl, X_test_nsl)
p_cnn     = predict_proba_cnn(cnn_nsl, X_test_nsl)
p_tf      = predict_proba_transformer(transformer_nsl, X_test_nsl)
p_vae     = predict_proba_vae(enc_nsl, dec_nsl, X_test_nsl)

proba_test_nsl = collect_probabilities(
    bilstm_nsl, cnn_nsl, transformer_nsl, enc_nsl, dec_nsl, X_test_nsl
)
p_ensemble = ensemble_nsl.predict_proba(proba_test_nsl)
y_pred_ens = ensemble_nsl.predict(proba_test_nsl)

# Compute all metrics
threshold = 0.5
all_metrics_nsl = []
for name, proba in [('BiLSTM', p_bilstm), ('CNN', p_cnn),
                    ('Transformer', p_tf), ('VAE', p_vae)]:
    preds = (proba >= threshold).astype(int)
    m = full_evaluation(y_test_nsl, preds, proba, name, 'NSL-KDD', out_nsl)
    all_metrics_nsl.append(m)

m_ens = full_evaluation(y_test_nsl, y_pred_ens, p_ensemble,
                        'Ensemble (Ours)', 'NSL-KDD', out_nsl)
all_metrics_nsl.append(m_ens)

# Plots
plot_roc_curves(
    [{'name':'BiLSTM','proba':p_bilstm}, {'name':'CNN','proba':p_cnn},
     {'name':'Transformer','proba':p_tf}, {'name':'VAE','proba':p_vae},
     {'name':'Ensemble','proba':p_ensemble}],
    y_test_nsl, out_nsl, 'NSL-KDD'
)
plot_metrics_bar(all_metrics_nsl, out_nsl, 'NSL-KDD')
for name, hist in [('BiLSTM', hist_bilstm_nsl), ('CNN', hist_cnn_nsl),
                   ('Transformer', hist_tf_nsl)]:
    plot_training_history(hist.history, out_nsl, f'{name}_NSL')

save_metrics_table(all_metrics_nsl, out_nsl, 'NSL-KDD')
print('\nNSL-KDD evaluation complete.')

In [ ]:
# ── Cell 12: Train all models on CICIDS2017 ───────────────────────────────
X_tr_c, X_vl_c, y_tr_c, y_vl_c = train_test_split(
    X_train_cic, y_train_cic, test_size=0.1,
    stratify=y_train_cic, random_state=42
)

print('\n===== Training BiLSTM (CICIDS2017) =====')
bilstm_cic, hist_bilstm_cic = train_bilstm(
    X_tr_c, y_tr_c, X_vl_c, y_vl_c,
    **{k:v for k,v in bilstm_params.items() if k != 'batch_size'},
    batch_size=bilstm_params.get('batch_size', 256), epochs=100,
    save_path=os.path.join(MODELS_DIR, 'bilstm_cic.keras')
)

print('\n===== Training CNN (CICIDS2017) =====')
cnn_cic, hist_cnn_cic = train_cnn(
    X_tr_c, y_tr_c, X_vl_c, y_vl_c,
    **{k:v for k,v in cnn_params.items() if k != 'batch_size'},
    batch_size=cnn_params.get('batch_size', 256), epochs=100,
    save_path=os.path.join(MODELS_DIR, 'cnn_cic.keras')
)

print('\n===== Training Transformer (CICIDS2017) =====')
transformer_cic, hist_tf_cic = train_transformer(
    X_tr_c, y_tr_c, X_vl_c, y_vl_c,
    **tf_params, epochs=100,
    save_path=os.path.join(MODELS_DIR, 'transformer_cic.keras')
)

print('\n===== Training VAE (CICIDS2017) =====')
vae_cic, enc_cic, dec_cic, hist_vae_cic = train_vae(
    X_tr_c, y_tr_c, **vae_params, epochs=100,
    save_dir=os.path.join(MODELS_DIR, 'vae_cic')
)

# Ensemble
proba_val_cic = collect_probabilities(
    bilstm_cic, cnn_cic, transformer_cic, enc_cic, dec_cic, X_vl_c
)
ensemble_cic = EnsembleMetaClassifier()
ensemble_cic.fit(proba_val_cic, y_vl_c,
                 model_names=['BiLSTM','CNN','Transformer','VAE'])
ensemble_cic.save(os.path.join(MODELS_DIR, 'ensemble_cic'))
print('\nAll CICIDS2017 models trained.')

In [ ]:
# ── Cell 13: Evaluate on CICIDS2017 ──────────────────────────────────────
out_cic = os.path.join(OUTPUTS_DIR, 'cicids')
os.makedirs(out_cic, exist_ok=True)

p_b_c = predict_proba_bilstm(bilstm_cic, X_test_cic)
p_c_c = predict_proba_cnn(cnn_cic, X_test_cic)
p_t_c = predict_proba_transformer(transformer_cic, X_test_cic)
p_v_c = predict_proba_vae(enc_cic, dec_cic, X_test_cic)

proba_test_cic = collect_probabilities(
    bilstm_cic, cnn_cic, transformer_cic, enc_cic, dec_cic, X_test_cic
)
p_ens_c  = ensemble_cic.predict_proba(proba_test_cic)
y_pred_c = ensemble_cic.predict(proba_test_cic)

all_metrics_cic = []
for name, proba in [('BiLSTM', p_b_c), ('CNN', p_c_c),
                    ('Transformer', p_t_c), ('VAE', p_v_c)]:
    preds = (proba >= 0.5).astype(int)
    m = full_evaluation(y_test_cic, preds, proba, name, 'CICIDS2017', out_cic)
    all_metrics_cic.append(m)

m_ens_c = full_evaluation(y_test_cic, y_pred_c, p_ens_c,
                          'Ensemble (Ours)', 'CICIDS2017', out_cic)
all_metrics_cic.append(m_ens_c)

plot_roc_curves(
    [{'name':'BiLSTM','proba':p_b_c}, {'name':'CNN','proba':p_c_c},
     {'name':'Transformer','proba':p_t_c}, {'name':'VAE','proba':p_v_c},
     {'name':'Ensemble','proba':p_ens_c}],
    y_test_cic, out_cic, 'CICIDS2017'
)
plot_metrics_bar(all_metrics_cic, out_cic, 'CICIDS2017')
save_metrics_table(all_metrics_cic, out_cic, 'CICIDS2017')
print('\nCICIDS2017 evaluation complete.')

In [ ]:
# ── Cell 14: Print final summary table ───────────────────────────────────
import pandas as pd
print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print('\n--- NSL-KDD ---')
df_nsl = pd.DataFrame(all_metrics_nsl)[['Model','Accuracy','Precision','Recall','F1-Score','AUC-ROC','FPR']]
print(df_nsl.to_string(index=False))
print('\n--- CICIDS2017 ---')
df_cic = pd.DataFrame(all_metrics_cic)[['Model','Accuracy','Precision','Recall','F1-Score','AUC-ROC','FPR']]
print(df_cic.to_string(index=False))
print('\nAll outputs saved to:', OUTPUTS_DIR)